In [ ]:
import json
import os
import re
from pathlib import Path

import pandas as pd

RUN_DIR = Path("runs/bomberman_ppo_20260907_021835")
LOG_DIR = RUN_DIR / "logs"


In [ ]:
def eval_from_checkpoints(run_dir):
    rows = []
    for meta_path in sorted((run_dir / "checkpoints").glob("checkpoint_*/metadata.json")):
        meta = json.loads(meta_path.read_text())
        suite = meta.get("eval_suite") or {}
        for case, value in suite.items():
            name = case.split(".")[1] if case.startswith("agent_code.") else case
            row = {"timesteps": meta["timesteps"], "case": name}
            if isinstance(value, dict):
                row.update(value)
            else:
                row["reward"] = value
            rows.append(row)
    return pd.DataFrame(rows)


checkpoint_df = eval_from_checkpoints(RUN_DIR)
checkpoint_df


In [ ]:
LINE = re.compile(
    r"eval (?:vs (?P<bot>\w+)(?: \([^)]*\))?|generalization \[(?P<gen>[^\]]+)\]):\s*(?P<body>.*?)\s*\(n=(?P<n>\d+)\)"
)
FIELD = re.compile(r"(?P<key>score|win|survived|length|reward)\s+(?P<value>[-+]?\d+(?:\.\d+)?)%?")
TIMESTEP = re.compile(r"Eval game finished at\s*(?P<timesteps>\d+)\s*timesteps")


def parse_eval_line(line):
    match = LINE.search(line)
    if not match:
        return None
    case = match.group("bot") or "gen_" + match.group("gen").replace(", ", "_").replace(" ", "")
    body = match.group("body")
    fields = {m.group("key"): float(m.group("value")) for m in FIELD.finditer(body)}
    if not fields:
        fields = {"reward": float(body)}
    for key in ("win", "survived"):
        if key in fields:
            fields[key] /= 100.0
    return case, fields, int(match.group("n"))


def eval_from_logs(log_dir):
    rows = []
    for log_path in sorted(log_dir.glob("*.out")):
        pending = []
        for line in log_path.read_text(errors="replace").splitlines():
            parsed = parse_eval_line(line)
            if parsed:
                pending.append(parsed)
                continue
            done = TIMESTEP.search(line)
            if done and pending:
                for case, fields, n in pending:
                    rows.append({"timesteps": int(done.group("timesteps")), "case": case, "n": n,
                                 "log_file": log_path.name, **fields})
                pending = []
    return pd.DataFrame(rows).drop_duplicates(["timesteps", "case"]).sort_values(["timesteps", "case"])


log_df = eval_from_logs(LOG_DIR) if LOG_DIR.exists() else pd.DataFrame()
log_df


In [ ]:
df = checkpoint_df if len(checkpoint_df) else log_df
metric = "score" if "score" in df.columns and df["score"].notna().any() else "reward"
table = df.pivot_table(index="timesteps", columns="case", values=metric)
table


In [ ]:
import matplotlib.pyplot as plt

ax = table.plot(figsize=(10, 5), marker=".")
ax.set_xlabel("timesteps")
ax.set_ylabel(metric)
ax.set_title(f"{RUN_DIR.name}: evaluation {metric} per case")
ax.grid(alpha=0.3)
plt.show()
